# Part 2 — Descriptive EDA

Issue [#6]: Revenue theo ngày/tuần/tháng/năm · Top category/segment/product · Region & city · Acquisition channel · Web traffic trend.

Mọi figure đều có title, axis label, legend.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data_loader import (
    build_line_items, build_daily_sales,
    load_customers, load_web_traffic, load_sales,
)

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.1)
FIG_DIR = '../reports/figures'

def save(fig, name):
    fig.savefig(f'{FIG_DIR}/{name}.png', dpi=150, bbox_inches='tight')
    plt.show()

li    = build_line_items()
daily = build_daily_sales()
sales = load_sales()
cust  = load_customers()
wt    = load_web_traffic()

print('line_items :', li.shape)
print('daily sales:', daily.shape)
print('date range :', daily['date'].min().date(), '→', daily['date'].max().date())

---
## 1 · Revenue theo ngày / tháng / năm

In [ ]:
# ── 1a. Daily revenue (rolling 30-day) ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily['date'], daily['revenue'].rolling(30).mean(), lw=1.5, label='30-day MA')
ax.fill_between(daily['date'], daily['revenue'], alpha=0.15, label='Daily')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.set_title('Daily Revenue (2012–2022) with 30-day Moving Average')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue (VND)')
ax.legend()
save(fig, '01a_daily_revenue')

In [ ]:
# ── 1b. Monthly revenue bar ──────────────────────────────────────────────────
monthly = daily.set_index('date').resample('ME')['revenue'].sum().reset_index()
monthly['year']  = monthly['date'].dt.year
monthly['month'] = monthly['date'].dt.month

fig, ax = plt.subplots(figsize=(14, 4))
pivot = monthly.pivot(index='month', columns='year', values='revenue') / 1e6
pivot.plot(ax=ax, marker='o', lw=1.5)
ax.set_title('Monthly Revenue by Year (VND M)')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (M VND)')
ax.set_xticks(range(1, 13))
ax.legend(title='Year', bbox_to_anchor=(1, 1))
save(fig, '01b_monthly_revenue_by_year')

In [ ]:
# ── 1c. Annual revenue bar ───────────────────────────────────────────────────
annual = daily.set_index('date').resample('YE')['revenue'].sum().reset_index()
annual['year'] = annual['date'].dt.year

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(annual['year'], annual['revenue'] / 1e9, color=sns.color_palette()[0])
ax.bar_label(bars, fmt='%.1fB', padding=3, fontsize=9)
ax.set_title('Annual Revenue 2012–2022')
ax.set_xlabel('Year')
ax.set_ylabel('Revenue (B VND)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}B'))
save(fig, '01c_annual_revenue')

---
## 2 · Top 10 Category / Segment / Product

In [ ]:
# ── 2a. Top 10 category by revenue ──────────────────────────────────────────
cat_rev = li.groupby('category')['line_revenue'].sum().nlargest(10).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(cat_rev.index, cat_rev.values / 1e9)
ax.bar_label(bars, fmt='%.1fB', padding=3, fontsize=9)
ax.set_title('Top 10 Product Categories by Total Revenue')
ax.set_xlabel('Revenue (B VND)')
ax.set_ylabel('Category')
save(fig, '02a_top10_category_revenue')

In [ ]:
# ── 2b. Segment revenue + gross margin ──────────────────────────────────────
seg = (
    li.groupby('segment')
    .agg(revenue=('line_revenue', 'sum'), cogs=('line_cogs', 'sum'))
    .sort_values('revenue', ascending=False)
    .reset_index()
)
seg['gross_margin'] = (seg['revenue'] - seg['cogs']) / seg['revenue']

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()
ax1.bar(seg['segment'], seg['revenue'] / 1e9, color=sns.color_palette()[0], alpha=0.8, label='Revenue')
ax2.plot(seg['segment'], seg['gross_margin'], 'o--', color='tomato', lw=1.8, label='Gross Margin')
ax1.set_title('Revenue & Gross Margin by Segment')
ax1.set_xlabel('Segment')
ax1.set_ylabel('Revenue (B VND)')
ax2.set_ylabel('Gross Margin')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
save(fig, '02b_segment_revenue_margin')

In [ ]:
# ── 2c. Top 10 products by revenue ──────────────────────────────────────────
prod_rev = (
    li.groupby(['product_id', 'product_name'])['line_revenue']
    .sum()
    .nlargest(10)
    .reset_index()
    .sort_values('line_revenue')
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(prod_rev['product_name'], prod_rev['line_revenue'] / 1e6)
ax.set_title('Top 10 Products by Total Revenue')
ax.set_xlabel('Revenue (M VND)')
ax.set_ylabel('Product')
save(fig, '02c_top10_product_revenue')

---
## 3 · Phân bố Region & City theo đơn và doanh thu

In [ ]:
# ── 3a. Revenue & order count by region ─────────────────────────────────────
region = (
    li.groupby('region')
    .agg(revenue=('line_revenue', 'sum'), orders=('order_id', 'nunique'))
    .sort_values('revenue', ascending=False)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(region['region'], region['revenue'] / 1e9, color=sns.color_palette()[:len(region)])
axes[0].set_title('Total Revenue by Region')
axes[0].set_ylabel('Revenue (B VND)')
axes[0].set_xlabel('Region')

axes[1].bar(region['region'], region['orders'], color=sns.color_palette()[:len(region)])
axes[1].set_title('Total Orders by Region')
axes[1].set_ylabel('# Orders')
axes[1].set_xlabel('Region')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

fig.suptitle('Regional Performance Overview', fontweight='bold')
fig.tight_layout()
save(fig, '03a_region_revenue_orders')

In [ ]:
# ── 3b. Top 10 cities by revenue ─────────────────────────────────────────────
city_rev = (
    li.groupby('geo_city')['line_revenue']
    .sum()
    .nlargest(10)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(city_rev.index, city_rev.values / 1e9)
ax.set_title('Top 10 Cities by Total Revenue')
ax.set_xlabel('Revenue (B VND)')
ax.set_ylabel('City')
save(fig, '03b_top10_city_revenue')

---
## 4 · Customer Acquisition Channel theo thời gian

In [ ]:
# ── 4a. New signups per year by channel ─────────────────────────────────────
cust2 = cust.dropna(subset=['acquisition_channel']).copy()
cust2['signup_year'] = cust2['signup_date'].dt.year

signup_channel = (
    cust2.groupby(['signup_year', 'acquisition_channel'])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(12, 5))
signup_channel.plot(kind='bar', stacked=True, ax=ax, colormap='tab10')
ax.set_title('New Customer Signups by Acquisition Channel and Year')
ax.set_xlabel('Year')
ax.set_ylabel('# New Customers')
ax.legend(title='Channel', bbox_to_anchor=(1, 1))
ax.tick_params(axis='x', rotation=45)
save(fig, '04a_signup_by_channel_year')

In [ ]:
# ── 4b. Channel mix pie ──────────────────────────────────────────────────────
channel_total = cust2['acquisition_channel'].value_counts()

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(
    channel_total,
    labels=channel_total.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette('tab10', len(channel_total)),
)
ax.set_title('Customer Acquisition Channel Mix (Overall)')
save(fig, '04b_channel_mix_pie')

---
## 5 · Web Traffic — Sessions & Unique Visitors xu hướng

In [ ]:
# ── 5a. Monthly sessions & unique visitors ───────────────────────────────────
wt2 = wt.set_index('date').resample('ME')[['sessions', 'unique_visitors']].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(wt2['date'], wt2['sessions'] / 1e3, label='Sessions (K)', lw=1.5)
ax.plot(wt2['date'], wt2['unique_visitors'] / 1e3, label='Unique Visitors (K)', lw=1.5, ls='--')
ax.set_title('Monthly Web Traffic: Sessions vs. Unique Visitors')
ax.set_xlabel('Date')
ax.set_ylabel('Count (K)')
ax.legend()
save(fig, '05a_monthly_web_traffic')

In [ ]:
# ── 5b. Avg bounce rate by traffic source ────────────────────────────────────
bounce = (
    wt.groupby('traffic_source')['bounce_rate']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(bounce['traffic_source'], bounce['bounce_rate'], color=sns.color_palette()[:len(bounce)])
ax.set_title('Average Bounce Rate by Traffic Source')
ax.set_xlabel('Traffic Source')
ax.set_ylabel('Avg Bounce Rate')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.tick_params(axis='x', rotation=30)
save(fig, '05b_bounce_rate_by_source')

---
## Key Findings (Descriptive Level)

Điền sau khi chạy notebook:

| # | Finding | Số liệu |
|---|---------|--------|
| 1 | Revenue tăng trưởng CAGR ... | ...% |
| 2 | Top category chiếm ...% tổng doanh thu | ... |
| 3 | Region X dẫn đầu doanh thu | ... |
| 4 | Channel X chiếm ...% acquisition | ... |
| 5 | Web traffic tăng mạnh từ năm ... | ... |